In [155]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from config import PATH_RAW_DATA, CSV_ENCODING, CSV_SEPARATOR, NA_MARKERS

pd.set_option('display.max_columns', None)

# 1.Extract

In [156]:
df = pd.read_csv(PATH_RAW_DATA, encoding=CSV_ENCODING, sep=CSV_SEPARATOR, na_values=NA_MARKERS)

# 2.Transform

- Trata coluna "nome_cientifico" pois alguns valores possuem "." ao final
- Substituir vírgula por pontos
- Utilizar a coluna "nome_cientifico" como "genero" utilizando split()
- Alterar linha 264(id_especie=202) especie deve ser "Pouteria"
- Alterar dtype das colunas numericas

In [157]:
# Colunas de identificação (não entram no cálculo de similaridade, só como chave/rótulo)
id_cols = [
    'id_especie',
    'nome_cientifico',
    'nome_popular_1',
    'genero',
    'especie',
    'familia'
]

# Núcleo físico-mecânico (baixa taxa de missing, base da similaridade)
numeric_features = [
    'densidade_basica',
    'contracao_tangencial',
    'contracao_radial',
    'contracao_volumetrica',
    'relacao_tangencial_radial',
    'flexao_seca_moe',
    'flexao_seca_mor',
    'compressao_paralela_seca',
    'dureza_janka_paralela_seca',
    'dureza_janka_transversal_seca',
    'cisalhamento_seca'
]

# Estético/sensorial (categóricas, relevantes se o uso-alvo for móveis/acabamento)
categorical_features = [
    'cor_cerne_classificacao',
    'textura',
    'gra',
    'brilho',
    'cerne_alburno',
    'figura_tangencial'
]

# Opcional: viabilidade de processamento (mais missing, usar com cautela/imputação)
processing_features = [
    'secagem_duracao_dias'
]

# Array combinado para o dataset de trabalho
colunas_selecionadas = id_cols + numeric_features + categorical_features + processing_features

df = df[colunas_selecionadas]
df.head(1)

,id_especie,nome_cientifico,nome_popular_1,genero,especie,familia,densidade_basica,contracao_tangencial,contracao_radial,contracao_volumetrica,relacao_tangencial_radial,flexao_seca_moe,flexao_seca_mor,compressao_paralela_seca,dureza_janka_paralela_seca,dureza_janka_transversal_seca,cisalhamento_seca,cor_cerne_classificacao,textura,gra,brilho,cerne_alburno,figura_tangencial,secagem_duracao_dias
0,21,Astronium lecointei,MUIRACATIARA-RAJADA,Astronium,lecointei,Anacardiaceae,"0,75","7,2","4,1",11,"1,76","12,94","145,63","84,14","7688,43","8659,29","11,77",marrom,media,"ondulada, sem desenho",alto nas superfícies longitudinais,NaN,NaN,NaN


## 2.1 - Função para tratar coluna "nome_cientifico"

In [158]:
'''
Remove o "." ao final de alguns valores para padronização
'''
def normalize_col(df:pd.DataFrame) -> pd.DataFrame:
    mask = df["nome_cientifico"].str.endswith(".", na=False)
    
    if mask.any():
        df.loc[mask, "nome_cientifico"] = df.loc[mask, "nome_cientifico"].str.rstrip(".")
    else:
        print(df.loc[:, "nome_cientifico"])
    return df


## 2.2 - Vírgula por ponto nas colunas numéricas

In [159]:
def num_col(df:pd.DataFrame) -> pd.DataFrame:
    df[numeric_features] = df[numeric_features].replace(",",".", regex=True)
    df[numeric_features] = df[numeric_features].astype(float)
    return df

## 2.3 - Função para consertar coluna "gênero"

In [160]:
def nome_cient_col(df: pd.DataFrame) -> pd.DataFrame:
    df["genero"] = df["nome_cientifico"].str.split().str[0]
    df["especie"] = df["nome_cientifico"].str.split().str[1]
    return df

## 2.9 - Chamada das funções

In [161]:
normalize_col(df)
num_col(df)
nome_cient_col(df)
df.head()

,id_especie,nome_cientifico,nome_popular_1,genero,especie,familia,densidade_basica,contracao_tangencial,contracao_radial,contracao_volumetrica,relacao_tangencial_radial,flexao_seca_moe,flexao_seca_mor,compressao_paralela_seca,dureza_janka_paralela_seca,dureza_janka_transversal_seca,cisalhamento_seca,cor_cerne_classificacao,textura,gra,brilho,cerne_alburno,figura_tangencial,secagem_duracao_dias
0,21,Astronium lecointei,MUIRACATIARA-RAJADA,Astronium,lecointei,Anacardiaceae,0.75,7.2,4.1,11.0,1.76,12.94,145.63,84.14,7688.43,8659.29,11.77,marrom,media,"ondulada, sem desenho",alto nas superfícies longitudinais,NaN,NaN,NaN
1,24,Bagassa guianensis,TATAJUBA,Bagassa,guianensis,Moraceae,0.70,5.8,4.1,9.5,1.41,11.57,124.45,78.55,9875.32,7384.43,12.55,amarela,"média, com desenho em estrias nas superfícies ...",entrecruzada geralmente,alto nas superfícies longitudinais,distintos,NaN,NaN
2,28,Bowdichia nitida,SUCUPIRA,Bowdichia,nitida,Fabaceae,0.77,7.4,4.5,12.3,1.64,13.53,153.96,86.79,12758.48,11307.10,12.55,marrom,grossa com pouco desenho,direita a ligeiramente entrecruzada,alto,distintos,NaN,NaN
3,47,Calophyllum brasiliense,JACAREÚBA,Calophyllum,brasiliense,Calophyllaceae,0.54,8.4,5.4,12.9,1.56,8.53,87.67,53.25,7864.95,5668.26,10.59,marrom,média e homogênea,"irregular e, geralmente, entrecruzada",moderado,pouco distintos a indistintos,NaN,NaN
4,50,Carapa guianensis,ANDIROBA,Carapa,guianensis,Meliaceae,0.56,7.0,4.6,11.8,1.52,10.30,94.83,53.45,8080.70,6295.89,9.61,marrom,média,direita a entrecruzada,fraco,indistintos,pouco desenho,NaN


# 3.Load

# restart

In [163]:
#%reset -f